In [41]:
# se verifica que los archivos de las clases existan en el entorno
!ls -l hf_loader.py dataset_summarizer.py

-rw-r--r-- 1 root root 2185 Oct 18 20:38 dataset_summarizer.py
-rw-r--r-- 1 root root 1461 Oct 18 20:30 hf_loader.py


In [42]:
# 1. Importar las clases desde los archivos .py
from hf_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer
import pandas as pd
import numpy as np

# 2. Obtener el DataFrame original desde Hugging Face
loader = HFDatasetLoader(dataset_name="scikit-learn/iris", split="train")
df_original = loader.load_data()

print(f"Dimensiones iniciales del DataFrame: {df_original.shape}")
display(df_original.head(3))

Dimensiones iniciales del DataFrame: (150, 6)


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa


In [43]:
# 1. copia y visualizacion del nombre de las columnas
df_transformado = df_original.copy()
cols_originales = df_transformado.columns.tolist()
print("Columnas originales detectadas:", cols_originales)

# ==========================================
# TRANSFORMACIÓN 1: Renombrar columnas de forma dinámica
# ==========================================
# se asignan nombres seguros basados en la posición de las columnas del dataset Iris
rename_dict = {
    cols_originales[0]: 'longitud_sepalo',
    cols_originales[1]: 'ancho_sepalo',
    cols_originales[2]: 'longitud_petalo',
    cols_originales[3]: 'ancho_petalo'
}
if len(cols_originales) > 4:
    rename_dict[cols_originales[4]] = 'especie'

df_transformado = df_transformado.rename(columns=rename_dict)

# ==========================================
# TRANSFORMACIÓN 2: Filtrar filas
# ==========================================
# se filtra usando la columna recién renombrada
df_transformado = df_transformado[df_transformado['longitud_petalo'] > 1.5]

# ==========================================
# TRANSFORMACIÓN 3: Crear columna calculada (Feature Engineering)
# ==========================================
df_transformado['area_petalo'] = df_transformado['longitud_petalo'] * df_transformado['ancho_petalo']

# ==========================================
# TRANSFORMACIÓN 4: Crear columna categórica condicional
# ==========================================
df_transformado['categoria_tamano'] = np.where(
    df_transformado['area_petalo'] > 5.0,
    'Grande',
    'Moderada'
)

# Resultado
print("\n¡Las 4 transformaciones se aplicaron con éxito!")
print(f"Dimensiones finales: {df_transformado.shape}")
display(df_transformado.head())

Columnas originales detectadas: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']

¡Las 4 transformaciones se aplicaron con éxito!
Dimensiones finales: (150, 8)


,longitud_sepalo,ancho_sepalo,longitud_petalo,ancho_petalo,especie,Species,area_petalo,categoria_tamano
0,1,5.1,3.5,1.4,0.2,Iris-setosa,4.90,Moderada
1,2,4.9,3.0,1.4,0.2,Iris-setosa,4.20,Moderada
2,3,4.7,3.2,1.3,0.2,Iris-setosa,4.16,Moderada
3,4,4.6,3.1,1.5,0.2,Iris-setosa,4.65,Moderada
4,5,5.0,3.6,1.4,0.2,Iris-setosa,5.04,Grande


In [44]:
print("=" * 60)
print("RESUMEN DE LAS 4 TRANSFORMACIONES APLICADAS:")
print("=" * 60)
print("1. Renombrar columnas: Se cambiaron los nombres originales en inglés a")
print("   español ('longitud_sepalo', 'longitud_petalo', etc.).")
print("2. Filtrado de filas: Se aplicó un filtro para conservar solamente los")
print("   registros con 'longitud_petalo' > 1.5 cm.")
print("3. Creación de columna calculada: Se generó una nueva columna numérica")
print("   llamada 'area_petalo' multiplicando longitud por ancho del pétalo.")
print("4. Creación de columna categórica: Se agregó una columna de texto")
print("   ('categoria_tamano') clasificando las flores según su área mediante una condición.")
print("=" * 60)

RESUMEN DE LAS 4 TRANSFORMACIONES APLICADAS:
1. Renombrar columnas: Se cambiaron los nombres originales en inglés a
   español ('longitud_sepalo', 'longitud_petalo', etc.).
2. Filtrado de filas: Se aplicó un filtro para conservar solamente los
   registros con 'longitud_petalo' > 1.5 cm.
3. Creación de columna calculada: Se generó una nueva columna numérica
   llamada 'area_petalo' multiplicando longitud por ancho del pétalo.
4. Creación de columna categórica: Se agregó una columna de texto
   ('categoria_tamano') clasificando las flores según su área mediante una condición.


In [45]:
# 4. Pasar el DataFrame transformado a la segunda clase para generar el archivo .txt
summarizer = DatasetSummarizer(df=df_transformado)
summarizer.save_summary_to_txt(filename="iris_4_transformaciones_summary.txt")

print("\n¡Archivo '.txt' generado exitosamente con el resumen de los datos transformados!")

¡Éxito! El archivo 'iris_4_transformaciones_summary.txt' ha sido generado correctamente.

¡Archivo '.txt' generado exitosamente con el resumen de los datos transformados!
